<a href="https://colab.research.google.com/github/shikhachourey1992/Mateplotlib/blob/main/Fake_and_Real_newsdetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import string
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import recall_score
from sklearn.metrics import precision_score
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
fake=pd.read_csv('Fake.csv')
print(fake)
true=pd.read_csv('True.csv')
print(true)


                                                   title  \
0       Donald Trump Sends Out Embarrassing New Year’...   
1       Drunk Bragging Trump Staffer Started Russian ...   
2       Sheriff David Clarke Becomes An Internet Joke...   
3       Trump Is So Obsessed He Even Has Obama’s Name...   
4       Pope Francis Just Called Out Donald Trump Dur...   
...                                                  ...   
23476  McPain: John McCain Furious That Iran Treated ...   
23477  JUSTICE? Yahoo Settles E-mail Privacy Class-ac...   
23478  Sunnistan: US and Allied ‘Safe Zone’ Plan to T...   
23479  How to Blow $700 Million: Al Jazeera America F...   
23480  10 U.S. Navy Sailors Held by Iranian Military ...   

                                                    text      subject  \
0      Donald Trump just couldn t wish all Americans ...         News   
1      House Intelligence Committee Chairman Devin Nu...         News   
2      On Friday, it was revealed that former Milwauk...    

In [3]:
fake['label']=0
true['label']=1

In [4]:
df=pd.concat([fake,true],ignore_index=True)
print(df)

                                                   title  \
0       Donald Trump Sends Out Embarrassing New Year’...   
1       Drunk Bragging Trump Staffer Started Russian ...   
2       Sheriff David Clarke Becomes An Internet Joke...   
3       Trump Is So Obsessed He Even Has Obama’s Name...   
4       Pope Francis Just Called Out Donald Trump Dur...   
...                                                  ...   
44893  'Fully committed' NATO backs new U.S. approach...   
44894  LexisNexis withdrew two products from Chinese ...   
44895  Minsk cultural hub becomes haven from authorities   
44896  Vatican upbeat on possibility of Pope Francis ...   
44897  Indonesia to buy $1.14 billion worth of Russia...   

                                                    text    subject  \
0      Donald Trump just couldn t wish all Americans ...       News   
1      House Intelligence Committee Chairman Devin Nu...       News   
2      On Friday, it was revealed that former Milwauk...       New

In [5]:
#shaffal data mixing data
df=df.sample(frac=1,random_state=42)
df.reset_index(drop=True,inplace=True)
df=df[['title','text','label']]
print(df.head())

                                               title  \
0  Ben Stein Calls Out 9th Circuit Court: Committ...   
1  Trump drops Steve Bannon from National Securit...   
2  Puerto Rico expects U.S. to lift Jones Act shi...   
3   OOPS: Trump Just Accidentally Confirmed He Le...   
4  Donald Trump heads for Scotland to reopen a go...   

                                                text  label  
0  21st Century Wire says Ben Stein, reputable pr...      0  
1  WASHINGTON (Reuters) - U.S. President Donald T...      1  
2  (Reuters) - Puerto Rico Governor Ricardo Rosse...      1  
3  On Monday, Donald Trump once again embarrassed...      0  
4  GLASGOW, Scotland (Reuters) - Most U.S. presid...      1  


In [6]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44898 entries, 0 to 44897
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   title   44898 non-null  object
 1   text    44898 non-null  object
 2   label   44898 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 1.0+ MB
None


In [7]:
print(df.isnull().sum())

title    0
text     0
label    0
dtype: int64


In [8]:
df.dropna(inplace=True)

In [9]:
#clean data
df['news']=df['title']+df['text']

def clean_text(text):
  text=text.lower()
  text=text.translate(str.maketrans('','',string.punctuation))
  return text

df['news']=df['news'].apply(clean_text)

x=df['news']
y=df['label']

vectorizer=TfidfVectorizer(
    stop_words='english',
    max_features=5000,
)

x=vectorizer.fit_transform(x)
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

model=LogisticRegression()
model.fit(x_train,y_train)
prediction=model.predict(x_test)
print(prediction)

[0 1 1 ... 0 1 1]


In [10]:
ac=accuracy_score(y_test,prediction)
print(ac)

0.9834075723830735


In [11]:
cm=confusion_matrix(y_test,prediction)
print(cm)

[[4614   96]
 [  53 4217]]


In [12]:

ps=precision_score(y_test,prediction)
print(ps)

0.9777417111059588


In [13]:
f1_s=f1_score(y_test,prediction)
print(f1_s)

0.9826401025282535


In [14]:
re_s=recall_score(y_test,prediction)
print(re_s)

0.9875878220140515


In [26]:
#PassiveAggressiveClassifier
from sklearn.linear_model import PassiveAggressiveClassifier
pac=PassiveAggressiveClassifier(max_iter=1000,random_state=42)
pac.fit(x_train,y_train)
prediction=pac.predict(x_test)
print(prediction)




[0 1 1 ... 0 1 1]


In [22]:
ac=accuracy_score(y_test,prediction)
print(ac)

0.989086859688196


In [23]:
ps=precision_score(y_test,prediction)
print(ps)

0.9882958801498127


In [24]:
cm=confusion_matrix(y_test,prediction)
print(cm)

[[4660   50]
 [  48 4222]]


In [19]:
f1_s=f1_score(y_test,prediction)
print(f1_s)

0.9892296885975181


In [20]:
re_s=recall_score(y_test,prediction)
print(re_s)

0.9894613583138173


In [37]:
news=[' The head of a conservative Republican faction in the U.S.']
news=[clean_text(i) for i in news]
news=vectorizer.transform(news)
prediction=pac.predict(news)
if prediction[0]==1:
  print('Realnews')
else:
  print('FakeNews')
print(prediction)



Realnews
[1]


In [38]:
df['label'].value_counts()


,count
label,
0,23481
1,21417


In [41]:
df['len']=df['news'].apply(len)
print(df.sort_values('len',ascending=False).head())

                                                   title  \
34710  MEDIA TRIPWIRE? Ping Pong Pizza Conspiracy Pro...   
10937  MEDIA TRIPWIRE? Ping Pong Pizza Conspiracy Pro...   
30806  ANTI-AMERICAN GEORGE SOROS Locks Arms With NFL...   
6016   ANTI-AMERICAN GEORGE SOROS Locks Arms With NFL...   
2529   The Las Vegas Mass Shooting – More to the Stor...   

                                                    text  label  \
34710   Funny how secrets travel. I d start to believ...      0   
10937   Funny how secrets travel. I d start to believ...      0   
30806  We just discovered another reason NOT to suppo...      0   
6016   We just discovered another reason NOT to suppo...      0   
2529   Shawn Helton 21st Century WireAlthough many ar...      0   

                                                    news    len  
34710  media tripwire ping pong pizza conspiracy prop...  50887  
10937  media tripwire ping pong pizza conspiracy prop...  50887  
30806  antiamerican george soros locks

In [42]:
print(df.groupby('label')['len'].mean())

label
0    2580.544270
1    2395.752626
Name: len, dtype: float64


In [43]:
from sklearn.feature_extraction.text import CountVectorizer

cv=CountVectorizer(
    stop_words='english'
)
x=cv.fit_transform(df['news'])
word_count=x.sum(axis=0)
words=cv.get_feature_names_out()
frequency=word_count.A1
result=list(zip(words,frequency))
result=sorted(result,key=lambda x:x[1],reverse=True)
print(result[:20])






[('trump', np.int64(140517)), ('said', np.int64(130255)), ('president', np.int64(53104)), ('people', np.int64(41751)), ('state', np.int64(33212)), ('new', np.int64(31823)), ('obama', np.int64(29900)), ('clinton', np.int64(29038)), ('house', np.int64(28717)), ('government', np.int64(27418)), ('reuters', np.int64(27355)), ('donald', np.int64(27354)), ('states', np.int64(26336)), ('just', np.int64(25829)), ('republican', np.int64(25284)), ('white', np.int64(23839)), ('told', np.int64(23541)), ('united', np.int64(23462)), ('like', np.int64(22482)), ('campaign', np.int64(21821))]


In [44]:
import joblib

joblib.dump(pac,"fake_news_model.pkl")

joblib.dump(vectorizer,"tfidf.pkl")

['tfidf.pkl']